# 🤖 nanoGPT — 從零實作

本 Notebook 實作一個完整可訓練的 GPT 語言模型，使用莎士比亞文本作為訓練資料。

**架構**：`Token Embedding + Positional Embedding → N × (MultiHead Attention + FFN) → LM Head`

**預估訓練時間**：CPU ~10分鐘 / GPU ~2分鐘（5000 steps）

## 1. 安裝與匯入套件

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import urllib.request
import os

print(f'PyTorch 版本: {torch.__version__}')
print(f'使用裝置: {"cuda" if torch.cuda.is_available() else "cpu"}')

## 2. 超參數設定

In [ ]:
# ── 訓練設定 ──────────────────────────────
batch_size    = 64      # 每次訓練的批次大小
block_size    = 256     # context 長度（最長看幾個 token）
max_iters     = 5000    # 訓練總步數
eval_interval = 500     # 每幾步評估一次
eval_iters    = 200     # 評估時平均幾個 batch
learning_rate = 3e-4

# ── 模型架構 ──────────────────────────────
n_embd  = 384   # embedding 維度
n_head  = 6     # attention heads 數量
n_layer = 6     # transformer block 層數
dropout = 0.2   # dropout 比例

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(1337)
print('超參數設定完成 ✓')

## 3. 資料載入與前處理

In [ ]:
# 下載莎士比亞資料集
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
if not os.path.exists('input.txt'):
    print('下載資料集...')
    urllib.request.urlretrieve(url, 'input.txt')

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(f'資料集大小: {len(text):,} 字元')
print(f'前 100 字元預覽:\n{text[:100]}')

In [ ]:
# ── 字元級 Tokenizer ──────────────────────
chars      = sorted(list(set(text)))
vocab_size = len(chars)
print(f'詞彙表大小: {vocab_size} 個唯一字元')
print(f'字元列表: {"".join(chars)}')

stoi = {ch: i for i, ch in enumerate(chars)}   # 字元 → 整數
itos = {i: ch for i, ch in enumerate(chars)}   # 整數 → 字元

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# 切分訓練 / 驗證集（90% / 10%）
data  = torch.tensor(encode(text), dtype=torch.long)
n     = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]
print(f'\n訓練集: {len(train_data):,} tokens')
print(f'驗證集: {len(val_data):,} tokens')

In [ ]:
def get_batch(split):
    """隨機取一個 batch，回傳 (輸入, 目標)"""
    data = train_data if split == 'train' else val_data
    ix   = torch.randint(len(data) - block_size, (batch_size,))
    x    = torch.stack([data[i:i+block_size]   for i in ix])
    y    = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

# 快速驗證
xb, yb = get_batch('train')
print(f'Input  shape: {xb.shape}  → (batch_size, block_size)')
print(f'Target shape: {yb.shape}  → (目標是 input 向右位移 1 格)')

## 4. 模型架構

In [ ]:
class Head(nn.Module):
    """單一 Causal Self-Attention Head"""

    def __init__(self, head_size):
        super().__init__()
        self.key     = nn.Linear(n_embd, head_size, bias=False)
        self.query   = nn.Linear(n_embd, head_size, bias=False)
        self.value   = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)
        # 下三角 mask：token 只能看自己和之前的位置
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)    # (B, T, head_size)
        q = self.query(x)  # (B, T, head_size)

        # Scaled dot-product attention
        wei = q @ k.transpose(-2, -1) * C**-0.5   # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))  # Causal mask
        wei = F.softmax(wei, dim=-1)               # 正規化
        wei = self.dropout(wei)

        v = self.value(x)
        return wei @ v   # (B, T, head_size)

print('Head 定義完成 ✓')

In [ ]:
class MultiHeadAttention(nn.Module):
    """多頭 Attention，並聯多個 Head 後投影"""

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads   = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj    = nn.Linear(n_embd, n_embd)   # 輸出投影
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # 每個 head 各自計算，concat 後投影回 n_embd
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))


class FeedForward(nn.Module):
    """Position-wise Feed-Forward（中間層擴大 4 倍）"""

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

print('MultiHeadAttention & FeedForward 定義完成 ✓')

In [ ]:
class Block(nn.Module):
    """Transformer Block：Attention → FFN，各加 Residual + LayerNorm"""

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa   = MultiHeadAttention(n_head, head_size)  # Self-Attention
        self.ffwd = FeedForward(n_embd)                    # FFN
        self.ln1  = nn.LayerNorm(n_embd)                   # Pre-Norm
        self.ln2  = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))    # Residual connection
        x = x + self.ffwd(self.ln2(x))  # Residual connection
        return x

print('Block 定義完成 ✓')

In [ ]:
class GPT(nn.Module):
    """完整 GPT 語言模型"""

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding    = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks  = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f    = nn.LayerNorm(n_embd)          # 最後的 LayerNorm
        self.lm_head = nn.Linear(n_embd, vocab_size) # 輸出 logits

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # Token + Position embedding
        tok_emb = self.token_embedding(idx)                                   # (B, T, C)
        pos_emb = self.position_embedding(torch.arange(T, device=device))     # (T, C)
        x = tok_emb + pos_emb                                                 # (B, T, C)

        x = self.blocks(x)   # N 層 Transformer
        x = self.ln_f(x)
        logits = self.lm_head(x)   # (B, T, vocab_size)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """自迴歸生成文字"""
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]         # 截取最後 block_size 個 token
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature # 只取最後一個時間步

            if top_k is not None:                   # Top-k sampling（可選）
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = float('-inf')

            probs    = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx      = torch.cat((idx, idx_next), dim=1)
        return idx


# 初始化模型
model = GPT(vocab_size).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'模型初始化完成 ✓')
print(f'總參數量: {total_params:,} ({total_params/1e6:.2f}M)')

## 5. 訓練

In [ ]:
@torch.no_grad()
def estimate_loss():
    """在 train / val 上各平均 eval_iters 個 batch 的 loss"""
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

print('評估函式定義完成 ✓')

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
train_losses, val_losses, steps_log = [], [], []

print(f'開始訓練（{max_iters} steps）...\n')
print(f'{"Step":>6}  {"Train Loss":>10}  {"Val Loss":>10}')
print('-' * 32)

for step in range(max_iters):

    # 定期評估
    if step % eval_interval == 0 or step == max_iters - 1:
        losses = estimate_loss()
        train_losses.append(losses['train'].item())
        val_losses.append(losses['val'].item())
        steps_log.append(step)
        print(f'{step:>6}  {losses["train"]:>10.4f}  {losses["val"]:>10.4f}')

    # 前向 + 反向
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print('\n訓練完成 ✓')

## 6. 視覺化訓練曲線

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 4))
plt.plot(steps_log, train_losses, label='Train Loss', marker='o', markersize=4)
plt.plot(steps_log, val_losses,   label='Val Loss',   marker='s', markersize=4)
plt.xlabel('Step')
plt.ylabel('Cross-Entropy Loss')
plt.title('nanoGPT 訓練曲線')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. 生成文字

In [ ]:
model.eval()

# 從空白 token 開始生成
context = torch.zeros((1, 1), dtype=torch.long, device=device)

generated = model.generate(
    context,
    max_new_tokens=500,
    temperature=0.8,   # 越低越保守；越高越有創意
    top_k=40           # 只從機率最高的 40 個 token 中抽樣
)

print('=== 生成結果 ===')
print(decode(generated[0].tolist()))

## 8. 儲存與載入模型

In [ ]:
# 儲存
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'train_losses': train_losses,
    'val_losses': val_losses,
    'vocab_size': vocab_size,
    'stoi': stoi,
    'itos': itos,
}, 'nanogpt_checkpoint.pt')
print('模型已儲存至 nanogpt_checkpoint.pt ✓')

# 載入（範例）
# checkpoint = torch.load('nanogpt_checkpoint.pt')
# model.load_state_dict(checkpoint['model_state_dict'])
# print('模型載入完成')

## 附錄：模型架構摘要

| 元件 | 說明 |
|------|------|
| Token Embedding | 將 token ID 映射到 `n_embd` 維向量 |
| Position Embedding | 注入位置資訊（learned，非 sinusoidal）|
| Causal Mask | 下三角 mask，確保 token 只看過去 |
| Pre-LayerNorm | 在 Attention/FFN 前正規化，穩定訓練 |
| Residual Connection | `x = x + sublayer(x)`，緩解梯度消失 |
| Scaled Attention | 除以 `√d_k`，防止 softmax 飽和 |

### 進一步改進方向
- 換用 **BPE Tokenizer**（如 tiktoken）取代字元級
- 換用 **RoPE** 位置編碼取代 learned PE
- 換用 **SwiGLU** 激活函式取代 ReLU
- 加入 **Flash Attention** 加速計算
- 使用更大的資料集（如 OpenWebText）